# Laboratorio 1 - AlpesHearth

## Integrantes

- Isabella Naranjo
- David Caro

## Exploración de los datos

Durante la fase de exploración de datos encontramos las siguientes irregularidades en el set de datos:
1. Existen 14 categorias que tienen registros faltantes, algunas de estas son la edad, el peso, la altura y el índice de masa corporal (BMI).
2. Las categorias booleanas vienen con strings como valor, es decir, "Y/N", en vez de True/False.
3. Existen registros de la fecha de servicio (Date of Service) que tienen formatos inconsistentes, es decir, vienen en múltiples formatos.
4. Existen algunos valores imposibles en las categorias de edad, peso, BMI, CVD Risk Score y Estimated LDL.
5. Existen 2 columnas de altura, una en metros y la otra en centimetros, esta información es la misma y puede ser redundante.

Ahora bien, para realizar una exploración más profunda utilizamos la librería pandas para entender la estructura general de los datos.

In [53]:
import pandas as pd 

df = pd.read_csv("../data/Datos Lab 1.csv")
print("Primeras 5 filas del DataFrame: ")
print(df.head())
print("\nUltimas 5 filas del DataFrame: ")
print(df.tail())
print("\nDimensiones del DataFrame")
print(f"Filas: {df.shape[0]}")
print(f"Columnas: {df.shape[1]}")
print(f"Total de celdas: {df.size}")

Primeras 5 filas del DataFrame: 
  Patient ID    Date of Service Sex   Age  Weight (kg)  Height (m)     BMI  \
0   isDx5313  November 08, 2023   M  44.0      114.300       1.720  38.600   
1   LHCK2961         20/03/2024   F  57.0       92.923       1.842  33.116   
2   WjVn1699         2021-05-27   F   NaN       73.400       1.650  27.000   
3   dCDO1109     April 18, 2022   F  35.0      113.300       1.780  35.800   
4   pnpE1080         01/11/2024   F  48.0      102.200       1.750  33.400   

   Abdominal Circumference (cm) Blood Pressure (mmHg)  \
0                       100.000                112/83   
1                       106.315                101/91   
2                        78.100                 90/74   
3                        79.600                 92/89   
4                       106.700                121/68   

   Total Cholesterol (mg/dL)  ...  Physical Activity Level  \
0                      228.0  ...                     High   
1                      158.0  .

In [54]:
df.info()
columnas_numericas = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
columnas_categoricas = df.select_dtypes(include=['object']).columns.tolist()
print(f"\nColumnas Numericas ({len(columnas_numericas)}): ")
print(columnas_numericas)
print(f"\nColumnas Categoricas ({len(columnas_categoricas)}): ")
print(columnas_categoricas)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1639 entries, 0 to 1638
Data columns (total 24 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   Patient ID                    1639 non-null   object 
 1   Date of Service               1639 non-null   object 
 2   Sex                           1639 non-null   object 
 3   Age                           1571 non-null   float64
 4   Weight (kg)                   1566 non-null   float64
 5   Height (m)                    1578 non-null   float64
 6   BMI                           1586 non-null   float64
 7   Abdominal Circumference (cm)  1578 non-null   float64
 8   Blood Pressure (mmHg)         1639 non-null   object 
 9   Total Cholesterol (mg/dL)     1571 non-null   float64
 10  HDL (mg/dL)                   1557 non-null   float64
 11  Fasting Blood Sugar (mg/dL)   1585 non-null   float64
 12  Smoking Status                1639 non-null   object 
 13  Dia

De estas líneas de código pudimos identificar lo siguiente respecto a la estructura de los datos:

1. Tenemos 1639 registros y 24 columnas (14 númericas y 10 categoricas).
2. Efectivamente hay valores nulos en las columnas de edad, peso, altura (m), BMI, circumferencia abdominal, colesterol total, HDL, glucosa en ayunas, altura (cm), relación cintura-altura, presión arterial sistolica, presión arterial diastólica, LDL estimado y puntaje de riesgo cardiovascular (CVD Risk Score).
3. Hay 29 valores nulos en nuestra variable objetivo **CVD Risk Score**.
4. La variable **Date of Service** aparece como tipo object dado que viene en diferentes formatos, sin embargo, fue documentada como Date y es necesario hacer un casting para poder manipularla.
5. Las variables **Systolic BP** y **Diastolic BP** son las componentes de la variable **Blood Pressure**, así que puede que podamos descarta dicha columna. 
6. La variable **CVD Risk Level** es categórica y probablemente sea derivada del **CVD Risk Score**, por lo que no será utilizada como variable predictora para evitar fuga de información.

Seguido a esto, corrimos las siguientes líneas de código, para identificar las irregularidades especificas de las diferentes categorias.

In [55]:
registros_duplicados = df.duplicated().sum()
print(f"Registros duplicados del data set: {registros_duplicados} ({(registros_duplicados * 100 / df.shape[0]).round(2)} %)")
duplicados_por_id = df.duplicated(subset=['Patient ID'])
print(f"Pacientes con ID duplicado: {duplicados_por_id.sum()}")
valores_nulos = df.isnull().sum()
porcentaje_valores_nulos = (df.isnull().sum() / len(df) * 100).round(2)
nulos = pd.DataFrame({
    'Valores Nulos': valores_nulos,
    'Porcentaje (%)': porcentaje_valores_nulos
})
print("Categorias con valores nulos y su respectivo porcentaje: ")
print(nulos[nulos['Valores Nulos'] > 0])
print("\nValores presión sanguínea: ", df["Blood Pressure (mmHg)"].unique()[:10])
print("\nValores máximos y mínimos de las variables clínicas:")
print("Edad: ", df["Age"].min(), ",", df["Age"].max())
print("Peso: ", df["Weight (kg)"].min(), ",", df["Weight (kg)"].max())
print("Altura: ", df["Height (m)"].min(), ",", df["Height (m)"].max())
print("Presión sistólica: ",df["Systolic BP"].min(), ",", df["Systolic BP"].max())
print("Presión diastólica: ", df["Diastolic BP"].min(), ",", df["Diastolic BP"].max())
print("CVD Risk Score: ", df["CVD Risk Score"].min(), ",", df["CVD Risk Score"].max())
print("Estimated LDL (mg/dL): ", df["Estimated LDL (mg/dL)"].min(), ",", df["Estimated LDL (mg/dL)"].max())


Registros duplicados del data set: 151 (9.21 %)
Pacientes con ID duplicado: 263
Categorias con valores nulos y su respectivo porcentaje: 
                              Valores Nulos  Porcentaje (%)
Age                                      68            4.15
Weight (kg)                              73            4.45
Height (m)                               61            3.72
BMI                                      53            3.23
Abdominal Circumference (cm)             61            3.72
Total Cholesterol (mg/dL)                68            4.15
HDL (mg/dL)                              82            5.00
Fasting Blood Sugar (mg/dL)              54            3.29
Height (cm)                              68            4.15
Waist-to-Height Ratio                    76            4.64
Systolic BP                              61            3.72
Diastolic BP                             85            5.19
Estimated LDL (mg/dL)                    57            3.48
CVD Risk Score        

Sobre estas nuevas líneas de código determinamos lo siguiente:
1. Hay 151 registros completamente duplicados, lo que representa el 9.21% del dataset, dichos registros duplicados deben ser eliminados para evitar sesgos en el entrenamiento del modelo.
2. Existen 263 pacientes con IDs duplicados, esto puede traducirse en diferentes escenarios como, registros completamente duplicados, múltiples visitas del paciente (diferente fecha), o error de identificación.
2. Como lo mencionamos anteriormente, la variable **Blood Pressure** es un string que combina las variables **Systolic BP** y **Diastolic BP**, así que esta variable es redundante y podríamos eliminarla.
3. Se observan valores decimales en la variable **edad** (6.13 años), cosa que podría indicar errores de registro o transformaciones previas en los datos.
4. El valor mínimo de la variable peso es de 13.26kg, el de edad es 6.13 años, el de BMI es de 4.3, el de CVD Risk Score y Estimated LDL es negativo, estos valores son imposibles dado que estamos hablando de un estudio cardiovascular de adultos.
5. El valor máximos de la presión sistólica fue de 202.711, dicho valor podría ser un outlier.
6. Hay valores nulos en múltiples variables clínicas, cosa que requiere estrategias de imputación en la fase de preparación de datos, adicionalmente, la variable objetivo CVD Risk Score presenta 1.77% de valores nulos, dichos valores deberan ser tratados con cuidado.

## Preparación de datos + Construcción de pipelines

Las principales irregularidades que encontramos en el set de datos fueron las siguientes:

1. 151 registros completamente duplicados.
2. 263 pacientes con IDs duplicados.
3. Categorias redundantes como **Blood Pressure (mmHg)**, **Height (cm)** y **CVD Risk Level**.
4. 29 valores nulos en nuestra variable objetivo **CVD Risk Score**.
5. Valores nulos en 14 columnas.
6. Valores imposibles en categorias como **Weight (kg)**, **Age**, **BMI**, **CVD Risk Score** y **Estimated LDL (mg/dL)**.
7. Tipos de datos erroneos respecto a la documentación, en especifico se espera que las categorias **Smoking Status**, **Diabetes Status** y **Family History of CVD** sean booleanos y no objects, de igual forma, **Date of Service** presenta multiples formatos, y se espera que sea tipo date y no object.

## Modelo 1. Linear Regression: Decisiones de limpieza de datos + pipeline/training

Dadas estas irregularidades, vamos a tomar las siguientes decisiones teniendo en cuenta que vamos a construir un modelo de regresión lineal:

1. Los registros completamente duplicados los vamos a eliminar, dejando únicamente la primera ocurrencia del registro.

In [56]:
df_clean = df.copy()
before = len(df_clean)
df_clean = df_clean.drop_duplicates(keep='first')
after = len(df_clean)

print(f"Registros eliminados: {before - after}")
print(f"Dimensiones actuales: {df_clean.shape}")

Registros eliminados: 151
Dimensiones actuales: (1488, 24)


2. Los pacientes con IDs duplicados los vamos a manejar de la siguiente manera, al estudiar los IDs duplicados nos dimos cuenta que todas las demás columnas son identicas en algunos casos, y en otros lo unico que cambia es el **CVD Risk Score**, en donde en algunas ocasiones es negativo, lo cual es un valor que no tiene sentido. Por tanto decidimos borrar el registro si el valor del CVD Risk Score es negativo, y únicamente quedarnos con la primera ocurrencia del ID, pues, en caso de que el valor no sea negativo, hacer un promedio basado en 2 CVD Risk Scores diferentes a pesar de que todas las columnas son iguales podría sesgar al modelo.

In [57]:
ids_duplicados = df_clean[df_clean['Patient ID'].duplicated(keep=False)]
print("Exploración de IDs duplicados: ")
print("\n")
print(ids_duplicados.sort_values('Patient ID').head(10))
df_clean = df_clean[df_clean['CVD Risk Score'] >= 0]
before_removing_duplicate_ids = len(df_clean['Patient ID'])
df_clean = df_clean.drop_duplicates(subset=['Patient ID'], keep='first')
after_removing_duplicate_ids = len(df_clean['Patient ID'])
print(f"\nRegistros eliminados (Patient ID): {before_removing_duplicate_ids - after_removing_duplicate_ids}")
print(f"Dimensiones actuales: {df_clean.shape}")

Exploración de IDs duplicados: 


     Patient ID Date of Service Sex   Age  Weight (kg)  Height (m)     BMI  \
17     AhYt1346      09-28-2020   M  41.0       71.300       1.730  23.800   
1227   AhYt1346      09-28-2020   M  41.0       71.300       1.730  23.800   
1117   Axab9332      2021-12-11   F  58.0       69.870       1.944  20.785   
383    Axab9332      2021-12-11   F  58.0       69.870       1.944  20.785   
1469   BQvQ6431      09/11/2020   M  33.0      118.300       1.690  41.400   
130    BQvQ6431      09/11/2020   M  33.0      118.300       1.690  41.400   
846    BqZp2317  April 15, 2025   F  72.0       57.836       1.554  24.008   
305    BqZp2317  April 15, 2025   F  72.0       57.836       1.554  24.008   
850    CDsa2651      23/06/2025   M  39.0       73.300       1.740  24.200   
1241   CDsa2651      23/06/2025   M  39.0       73.300       1.740  24.200   

      Abdominal Circumference (cm) Blood Pressure (mmHg)  \
17                         107.900             

3. Las categorias redundantes como **Blood Pressure (mmHg)**, **Height (cm)** y **CVD Risk Level** van a ser removidas, pues no se necesitan.

In [58]:
columns_to_drop = ['Patient ID', 'Blood Pressure (mmHg)', 'Height (cm)', 'CVD Risk Level']
columns_to_drop = [col for col in columns_to_drop if col in df_clean.columns]
df_clean = df_clean.drop(columns=columns_to_drop)
print("Verificar que las columnas fueron removidas: ")
print("\n")
print(df_clean.head())

Verificar que las columnas fueron removidas: 


     Date of Service Sex   Age  Weight (kg)  Height (m)     BMI  \
0  November 08, 2023   M  44.0      114.300       1.720  38.600   
1         20/03/2024   F  57.0       92.923       1.842  33.116   
2         2021-05-27   F   NaN       73.400       1.650  27.000   
3     April 18, 2022   F  35.0      113.300       1.780  35.800   
4         01/11/2024   F  48.0      102.200       1.750  33.400   

   Abdominal Circumference (cm)  Total Cholesterol (mg/dL)  HDL (mg/dL)  \
0                       100.000                      228.0         77.0   
1                       106.315                      158.0         71.0   
2                        78.100                      135.0         60.0   
3                        79.600                      158.0         34.0   
4                       106.700                      207.0         49.0   

   Fasting Blood Sugar (mg/dL) Smoking Status Diabetes Status  \
0                         91.0   

4. Los 29 valores nulos de la variable objetivo **CVD Risk Score** fueron removidos al eliminar los duplicados del set de datos.

In [59]:
valores_nulos_clean = df_clean.isnull().sum()
porcentaje_valores_nulos_clean = (df_clean.isnull().sum() / len(df) * 100).round(2)
nulos_clean = pd.DataFrame({
    'Valores Nulos': valores_nulos_clean,
    'Porcentaje (%)': porcentaje_valores_nulos_clean
})
print("Categorias con valores nulos y su respectivo porcentaje del set de datos de limpieza: ")
print(nulos_clean[nulos_clean['Valores Nulos'] > 0])

Categorias con valores nulos y su respectivo porcentaje del set de datos de limpieza: 
                              Valores Nulos  Porcentaje (%)
Age                                      58            3.54
Weight (kg)                              62            3.78
Height (m)                               48            2.93
BMI                                      42            2.56
Abdominal Circumference (cm)             46            2.81
Total Cholesterol (mg/dL)                56            3.42
HDL (mg/dL)                              66            4.03
Fasting Blood Sugar (mg/dL)              46            2.81
Waist-to-Height Ratio                    63            3.84
Systolic BP                              53            3.23
Diastolic BP                             59            3.60
Estimated LDL (mg/dL)                    47            2.87


5. Para manejar los valores nulos de las variables que aparecen en el recuadro superior, decidimos hacer una imputación con la mediana, puesto que este método es robusto a outliers (que están presentes en gran parte de las variables de nuestros datos). Por tanto, vamos a realizar la imputación dentro del pipeline, puesto que de esta manera evitamos usar datos que serán de test y evitamos que ocurra data leakage.

In [60]:
# Nulos imputados posteriormente con la mediana en el pipeline

6. Los valores imposibles en la variable objetivo (CVD Risk Score < 0) fueron eliminados. Para las demás variables con valores fisiológicamente imposibles (Age < 18, Weight < 30kg, BMI < 10, LDL < 0), decidimos convertirlos a nulos para ser imputados posteriormente con la mediana en el pipeline.

In [61]:
import numpy as np

df_clean.loc[df_clean['Age'] < 18, 'Age'] = np.nan
df_clean.loc[df_clean['Weight (kg)'] < 30, 'Weight (kg)'] = np.nan
df_clean.loc[df_clean['BMI'] < 10, 'BMI'] = np.nan
df_clean.loc[df_clean['Estimated LDL (mg/dL)'] < 0, 'Estimated LDL (mg/dL)'] = np.nan

print("Valores imposibles en df_clean:")
print(f"Age < 18: {len(df_clean[df_clean['Age'] < 18])}")
print(f"Weight (kg) < 30: {len(df_clean[df_clean['Weight (kg)'] < 30])}")
print(f"BMI < 10: {len(df_clean[df_clean['BMI'] < 10])}")
print(f"CVD Risk Score < 0: {len(df_clean[df_clean['CVD Risk Score'] < 0])}")
print(f"Estimated LDL (mg/dL) < 0: {len(df_clean[df_clean['Estimated LDL (mg/dL)'] < 0])}")

# Nulos imputados posteriormente con la mediana en el pipeline


Valores imposibles en df_clean:
Age < 18: 0
Weight (kg) < 30: 0
BMI < 10: 0
CVD Risk Score < 0: 0
Estimated LDL (mg/dL) < 0: 0


7. Las variables **Smoking Status**, **Diabetes Status** y **Family History of CVD** vienen como strings ('Y'/'N') en lugar de booleanos. Decidimos mantenerlas así ya que el OneHotEncoder del pipeline las transformará automáticamente. Finalmente decidimos eliminar la variable **Date of Service**, dado que no es un predictor relevante para el riesgo cardiovascular y presenta varios formatos inconsistentes.

In [62]:
if 'Date of Service' in df_clean.columns:
    df_clean = df_clean.drop(columns='Date of Service')
print("Verificar que ya no se encuentra la columna en los datos: ")
print("\n")
print(df_clean.head(1))

# Posterior implementación del OneHotEncoder en el pipeline para el manejo de las demás variables.

Verificar que ya no se encuentra la columna en los datos: 


  Sex   Age  Weight (kg)  Height (m)   BMI  Abdominal Circumference (cm)  \
0   M  44.0        114.3        1.72  38.6                         100.0   

   Total Cholesterol (mg/dL)  HDL (mg/dL)  Fasting Blood Sugar (mg/dL)  \
0                      228.0         77.0                         91.0   

  Smoking Status Diabetes Status Physical Activity Level  \
0              Y               Y                    High   

  Family History of CVD  Waist-to-Height Ratio  Systolic BP  Diastolic BP  \
0                     N                  0.581        112.0          83.0   

  Blood Pressure Category  Estimated LDL (mg/dL)  CVD Risk Score  
0    Hypertension Stage 1                  121.0           19.88  


### Construcción del pipeline

1. Vamos a definir nuestro target Y.

In [63]:
target = "CVD Risk Score"
X = df_clean.drop(columns=[target]) #todas las columnas que explican el target
y = df_clean[target]

2. División entrenamiento – test

In [64]:
from sklearn.model_selection import train_test_split # type: ignore

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)

3. Separación por tipo de variable

In [65]:
numeric_features = X.select_dtypes(include=["int64","float64"]).columns
categorical_features = X.select_dtypes(include=["object","category"]).columns

print("Numéricas:", list(numeric_features))
print("Categóricas:", list(categorical_features))


Numéricas: ['Age', 'Weight (kg)', 'Height (m)', 'BMI', 'Abdominal Circumference (cm)', 'Total Cholesterol (mg/dL)', 'HDL (mg/dL)', 'Fasting Blood Sugar (mg/dL)', 'Waist-to-Height Ratio', 'Systolic BP', 'Diastolic BP', 'Estimated LDL (mg/dL)']
Categóricas: ['Sex', 'Smoking Status', 'Diabetes Status', 'Physical Activity Level', 'Family History of CVD', 'Blood Pressure Category']


4. **Transfromación de datos:** imputación, onehot, scaler, preprocesador

In [66]:
from sklearn.pipeline import Pipeline # type: ignore
from sklearn.compose import ColumnTransformer # type: ignore
from sklearn.preprocessing import StandardScaler, OneHotEncoder # type: ignore
from sklearn.impute import SimpleImputer # type: ignore

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(drop='first', handle_unknown="ignore", sparse_output=False))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

5. **Entrenamiento:** En este literal entrenaremos el modelo usando los datos de entrenamiento, sacaremos unas metricas preliminares para ir viendo como se comporta, sin embargo, estas métricas no se usaran en la comparación de modelos. 

In [67]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np

pipeline_modelo1 = Pipeline(steps=[
    ("preprocesamiento", preprocessor),
    ("modelo", LinearRegression())
])

pipeline_modelo1.fit(X_train, y_train) #Entrenamiento

#Predicción 
y_train_pred = pipeline_modelo1.predict(X_train)
y_test_pred = pipeline_modelo1.predict(X_test)

print("Métricas sobre TRAIN:")
print(f"RMSE: {np.sqrt(mean_squared_error(y_train, y_train_pred)):.4f}")
print(f"MAE: {mean_absolute_error(y_train, y_train_pred):.4f}")
print(f"R²: {r2_score(y_train, y_train_pred):.4f}")

print("\nMétricas sobre TEST:")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, y_test_pred)):.4f}")
print(f"MAE: {mean_absolute_error(y_test, y_test_pred):.4f}")
print(f"R²: {r2_score(y_test, y_test_pred):.4f}")

Métricas sobre TRAIN:
RMSE: 10.1186
MAE: 3.0791
R²: 0.0666

Métricas sobre TEST:
RMSE: 10.9689
MAE: 3.2228
R²: 0.0404


De las métricas preliminares podemos decir que:

1. El modelo solo está explicando 6.6% de la variabilidad del CVD Risk Score.
2. El MAE es 3, lo que nos indica un error promedio razonable, sin embargo el RMSE fue 10.1186, así existen outliers inflando el RMSE.

## Modelo 2. Ridge: Decisiones de limpieza de datos + pipeline/training

Iniciaremos creando una copia del DataFrame original para evitar problemas, seguido a esto, vamos a eliminar todos los registros que tengan en nulo nuestra variable objetivo, así como todos los registros duplicados. 


In [68]:
df_model1 = df.copy()
df_model1 = df_model1.dropna(subset=["CVD Risk Score"])
before = len(df_model1)
df_model1 = df_model1.drop_duplicates(keep="first") #mantenemos solo la primera ocurrencia
after = len(df_model1)

print(f"Duplicados idénticos eliminados: {before - after}")

Duplicados idénticos eliminados: 150


Como lo vimos anteriormente, existen algunos registros con el Patient ID duplicado donde lo unico que cambia es el **CVD Risk Score**, esto lo vamos a manejar de la siguiente forma:
- Si el CVD Risk Score es negativo, eliminamos el registro.
- Si por el contrario, los duplicados tienen un CVD Risk Score positivo, solo vamos a tomar un registro y promediaremos la variable.

Anteriormente mencionamos que promediar la variable objetivo podría generar un sesgo, sin embargo, vamos a hacerlo para ver como se comporta el modelo.

In [69]:
print("Antes:", df_model1["Patient ID"].duplicated().sum(), "IDs duplicados")
ids_duplicados = df_model1[df_model1["Patient ID"].duplicated(keep=False)]
df_model1 = df_model1[df_model1["CVD Risk Score"] >= 0]

cols_sin_target = df_model1.columns.tolist()
cols_sin_target.remove("CVD Risk Score")
cols_sin_target.remove("Patient ID")
df_base = df_model1.drop_duplicates(subset=["Patient ID"], keep="first")[["Patient ID"] + cols_sin_target]
df_score_prom = df_model1.groupby("Patient ID")["CVD Risk Score"].mean().reset_index()
df_model1 = df_base.merge(df_score_prom, on="Patient ID", how="left")

#verificacion
print("Después:", df_model1["Patient ID"].duplicated().sum(), "IDs duplicados")
print(df_model1.shape)



Antes: 111 IDs duplicados
Después: 0 IDs duplicados
(1344, 24)


Seguido a esto, eliminaremos las mismas columnas que se eliminaron en el modelo 1, pues algunas son redundantes y otras no aportan nada al modelo.

In [70]:
columns_to_drop = [
    'Patient ID',
    'Blood Pressure (mmHg)',
    'Height (cm)',
    'CVD Risk Level',
    'Date of Service'
]

columns_to_drop = [col for col in columns_to_drop if col in df_model1.columns]
df_model1 = df_model1.drop(columns=columns_to_drop)

Ahora, como lo mencionamos anteriormente, en este modelo vamos a optar por eliminar todos los valores que no sean clínicamente convenientes/posibles, esto con el fin de ver como se comporta el modelo. Es decir, no asignaremos valores coherentes a dichos registros, simplemente no se usaran para entrenar el modelo.

In [71]:
df_model1 = df_model1[df_model1["Age"] >= 18]
df_model1 = df_model1[df_model1["Weight (kg)"] >= 30]
df_model1 = df_model1[df_model1["BMI"] >= 10]
df_model1 = df_model1[df_model1["Estimated LDL (mg/dL)"] >= 0]

print("Dimensiones después de eliminar valores imposibles:")
print(df_model1.shape)


Dimensiones después de eliminar valores imposibles:
(1119, 19)


Finalmente, vamos a eliminar los outliers extremos que no se hayan tenido en cuenta dentro del tratamiento de las variables clínicas, para asegurar que los datos de entrenamiento estén los más limpios y coherentes posible.

- Para llevar a cabo este proceso, utilizaremos un rango IQR, este  consiste en calcular el primer y el tercer cuartil de los datos, pues estos encapsulan el 50% central de los datos. A partir de estos valores se calcula el IQR, sin embargo, este nos da un rango pequeño, así que lo multiplicamos por 1.5 para ampliar el rango, cualquier registro que tenga un valor que caiga fuera de ese rango será eliminado, pues se puede considerar como outlier. 

In [72]:
numeric_cols = df_model1.select_dtypes(include=["int64", "float64"]).columns
numeric_cols = [c for c in numeric_cols if c != "CVD Risk Score"]
print(numeric_cols)

def quitar_outliers(df, col):
    Q1 = df[col].quantile(0.25)   
    Q3 = df[col].quantile(0.75)   
    IQR = Q3 - Q1                 
    
    lower = Q1 - 1.5 * IQR        
    upper = Q3 + 1.5 * IQR        
    
    df_filtrado = df[(df[col] >= lower) & (df[col] <= upper)]
    
    print(f"{col}: antes={len(df)}, después={len(df_filtrado)}")
    return df_filtrado
for col in numeric_cols:
    df_model1 = quitar_outliers(df_model1, col)

print("Dimensiones finales después de quitar outliers:")
print(df_model1.shape)


['Age', 'Weight (kg)', 'Height (m)', 'BMI', 'Abdominal Circumference (cm)', 'Total Cholesterol (mg/dL)', 'HDL (mg/dL)', 'Fasting Blood Sugar (mg/dL)', 'Waist-to-Height Ratio', 'Systolic BP', 'Diastolic BP', 'Estimated LDL (mg/dL)']
Age: antes=1119, después=1115
Weight (kg): antes=1115, después=1115
Height (m): antes=1115, después=1071
BMI: antes=1071, después=1065
Abdominal Circumference (cm): antes=1065, después=1028
Total Cholesterol (mg/dL): antes=1028, después=983
HDL (mg/dL): antes=983, después=930
Fasting Blood Sugar (mg/dL): antes=930, después=890
Waist-to-Height Ratio: antes=890, después=837
Systolic BP: antes=837, después=803
Diastolic BP: antes=803, después=757
Estimated LDL (mg/dL): antes=757, después=757
Dimensiones finales después de quitar outliers:
(757, 19)


### Pipeline + entrenamiento

Ahora vamos a iniciar con la construcción del pipeline para este modelo, primero vamos a separar X y Y.

In [73]:
target = "CVD Risk Score"

X = df_model1.drop(columns=[target])
y = df_model1[target]

Ahora hacemos la división entrenamiento test. 

In [74]:
from sklearn.model_selection import train_test_split # type: ignore

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.25,
    random_state=42
)

Seguido a esto, separamos por tipo de variable. 

In [75]:
numeric_features = X_train.select_dtypes(include=["int64", "float64"]).columns
categorical_features = X_train.select_dtypes(include=["object", "category"]).columns

print("Numéricas:", list(numeric_features))
print("Categóricas:", list(categorical_features))

Numéricas: ['Age', 'Weight (kg)', 'Height (m)', 'BMI', 'Abdominal Circumference (cm)', 'Total Cholesterol (mg/dL)', 'HDL (mg/dL)', 'Fasting Blood Sugar (mg/dL)', 'Waist-to-Height Ratio', 'Systolic BP', 'Diastolic BP', 'Estimated LDL (mg/dL)']
Categóricas: ['Sex', 'Smoking Status', 'Diabetes Status', 'Physical Activity Level', 'Family History of CVD', 'Blood Pressure Category']


**Transfromación de datos:** imputación, onehot, scaler, preprocesador

In [76]:
from sklearn.pipeline import Pipeline  # type: ignore
from sklearn.impute import SimpleImputer  # type: ignore
from sklearn.preprocessing import StandardScaler, OneHotEncoder  # type: ignore
from sklearn.compose import ColumnTransformer  # type: ignore

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(drop="first", handle_unknown="ignore", sparse_output=False))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

**Entrenamiento:** Finalmente entrenaremos el modelo usando los datos de entrenamiento, y sacaremos unas métricas preliminares para ir viendo como se comporta, teniendo en cuenta que estas métricas no se usaran en la comparación de modelos. 

In [77]:
from sklearn.linear_model import Ridge  # type: ignore
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score  # type: ignore
import numpy as np # type: ignore

pipeline_ridge = Pipeline(steps=[
    ("preprocesamiento", preprocessor),
    ("modelo", Ridge(alpha=1.0, random_state=42))
])

pipeline_ridge.fit(X_train, y_train) #entrenamiento

#prediccion
y_train_pred = pipeline_ridge.predict(X_train)
y_test_pred  = pipeline_ridge.predict(X_test)

print("Métricas sobre TRAIN (RIDGE):")
print(f"RMSE: {np.sqrt(mean_squared_error(y_train, y_train_pred)):.4f}")
print(f"MAE:  {mean_absolute_error(y_train, y_train_pred):.4f}")
print(f"R²:   {r2_score(y_train, y_train_pred):.4f}")
print("\nMétricas sobre TEST (RIDGE):")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, y_test_pred)):.4f}")
print(f"MAE:  {mean_absolute_error(y_test, y_test_pred):.4f}")
print(f"R²:   {r2_score(y_test, y_test_pred):.4f}")


Métricas sobre TRAIN (RIDGE):
RMSE: 10.9037
MAE:  4.0245
R²:   0.0946

Métricas sobre TEST (RIDGE):
RMSE: 8.7270
MAE:  3.6618
R²:   0.1129


### Modelo 3. ElasticNet: Decisiones de limpieza de datos + pipeline/training

En primera instancia, vamos a eliminar los registros nulos de nuestra variable objetivo y tambien aquellos registros que son completamente identicos.

In [78]:
df_model2 = df.copy()
df_model2 = df_model2.dropna(subset=["CVD Risk Score"])
before2 = len(df_model2)
df_model2 = df_model2.drop_duplicates(keep="first") #mantenemos solo la primera ocurrencia
after2 = len(df_model2)

print(f"Duplicados idénticos eliminados: {before2 - after2}")

Duplicados idénticos eliminados: 150


Posteriormente, para el manejo de los IDs repetidos vamos a dejar unicamente la primera ocurrencia, pues aquellos IDs que tienen un **CVD Risk Score** negativo  tambien son eliminados.

In [79]:
df_model2 = df_model2[df_model2['CVD Risk Score'] >= 0]
before_removing_duplicate_ids2 = len(df_model2['Patient ID'])
df_model2 = df_model2.drop_duplicates(subset=['Patient ID'], keep='first')
after_removing_duplicate_ids2 = len(df_model2['Patient ID'])
print(f"\nRegistros eliminados (Patient ID): {before_removing_duplicate_ids2 - after_removing_duplicate_ids2}")
print("Después:", df_model2["Patient ID"].duplicated().sum(), "IDs duplicados")
print(f"Dimensiones actuales: {df_model2.shape}")


Registros eliminados (Patient ID): 107
Después: 0 IDs duplicados
Dimensiones actuales: (1344, 24)


Las categorias **Patient ID**, **Blood Pressure (mmHg)**, **Height (cm)**, **CVD Risk Level** y **Date of Service** fueron removidas, pues son redundantes o no tienen estricta correlación con nuestra variable objetivo.

In [80]:
columns_to_drop_2 = ['Patient ID', 'Blood Pressure (mmHg)', 'Height (cm)', 'CVD Risk Level', 'Date of Service']
columns_to_drop_2 = [col2 for col2 in columns_to_drop_2 if col2 in df_model2.columns]
df_model2 = df_model2.drop(columns=columns_to_drop_2)
print("Verificar que las columnas fueron removidas: ")
print("\n")
print(df_model2.head())

Verificar que las columnas fueron removidas: 


  Sex   Age  Weight (kg)  Height (m)     BMI  Abdominal Circumference (cm)  \
0   M  44.0      114.300       1.720  38.600                       100.000   
1   F  57.0       92.923       1.842  33.116                       106.315   
2   F   NaN       73.400       1.650  27.000                        78.100   
3   F  35.0      113.300       1.780  35.800                        79.600   
4   F  48.0      102.200       1.750  33.400                       106.700   

   Total Cholesterol (mg/dL)  HDL (mg/dL)  Fasting Blood Sugar (mg/dL)  \
0                      228.0         77.0                         91.0   
1                      158.0         71.0                         76.0   
2                      135.0         60.0                        150.0   
3                      158.0         34.0                        111.0   
4                      207.0         49.0                        147.0   

  Smoking Status Diabetes Status Physi

En segunda instancia 